# Step 2 — Two variable building blocks

This experiment tests whether similarities calculated independently for two building-block positions predict the similarity of their combinatorially assembled products. The complete DNA-encoded library is retained: no identifier-level records, stereoisomers, or other isomers are removed.

For each descriptor, BB1 and BB2 similarities are combined using arithmetic and geometric means. These predicted similarities are compared with the fingerprint similarity of the assembled product. Spearman correlation and nearest-neighbor preservation are the primary measures; Pearson correlation, MAE, and RMSE are secondary measures.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import coaf
from validation.two_component_similarity import (
    TwoComponentConfig,
    load_two_component_dataset,
    run_two_component_benchmark,
)

PROJECT_ROOT = Path.cwd()
DATA_FILE = PROJECT_ROOT / 'data' / 'Combinatorial_FS_DEL.csv'
RESULT_DIR = PROJECT_ROOT / 'results' / 'step2'
FIGURE_DIR = RESULT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('COAF implementation:', coaf.__file__)

## Load and validate the complete library

The loader verifies the five-column schema, missing values, one-to-one identifier-to-SMILES mappings, complete BB1 × BB2 coverage, RDKit parsing, connectivity, and exactly one Hg marker per structure. It does **not** deduplicate structures.

In [ ]:
dataset = load_two_component_dataset(DATA_FILE)

dataset_overview = pd.DataFrame([{
    'Library products retained': dataset.n_products,
    'BB1 identifiers': len(dataset.bb1_table),
    'BB2 identifiers': len(dataset.bb2_table),
    'Expected Cartesian products': len(dataset.bb1_table) * len(dataset.bb2_table),
}])
display(dataset_overview)
display(dataset.frame[['product_id', 'source_row', 'BB1', 'BB2', 'SMILES_Product']].head())

The first table confirms complete combinatorial coverage. The second is the beginning of the product manifest: `product_id` is an analysis identifier and `source_row` points back to the unmodified source CSV.

## Run the full-library benchmark

A full 76,109 × 76,109 product matrix would contain approximately 2.9 billion unique pairs. Instead, the code deterministically shuffles the complete library and partitions it into balanced batches of approximately 1,000 products. Every product appears in exactly one batch, every descriptor uses identical batches, and all within-batch pairs are evaluated.

Fingerprint generation for all products can take several minutes. The result tables are saved under `results/step2`.

In [ ]:
config = TwoComponentConfig(
    n_bits=1024,
    ecfp_radius=3,
    coaf_radius=3,
    linear_max_path_length=6,
    neighbor_k=(5, 10),
    batch_size=1000,
    random_seed=123,
    aggregation_methods=('arithmetic_mean', 'geometric_mean'),
    representative_pair_sample_size=10_000,
)

results = run_two_component_benchmark(
    dataset,
    config=config,
    output_dir=RESULT_DIR,
)

## Dataset and batch audit

These outputs document that all original rows were retained and assigned exactly once. The batches are a memory-management device, not a filter or descriptor-specific sample.

In [ ]:
display(results.dataset_summary)

batch_audit = (
    results.batch_manifest.groupby('batch', as_index=False)
    .agg(n_products=('product_id', 'size'), unique_products=('product_id', 'nunique'))
)
display(batch_audit.describe().round(2))
assert len(results.batch_manifest) == dataset.n_products
assert results.batch_manifest['product_id'].nunique() == dataset.n_products

## Batch-level results

Each row corresponds to one descriptor, one aggregation rule, and one balanced product batch. `n_pairs` is the number of unique product pairs in that batch. Higher correlation and neighbor preservation are better; lower MAE and RMSE are better.

In [ ]:
display(results.per_batch_metrics.head(12).round(4))

## Plotting functions

The following functions control only presentation and therefore remain in the notebook. Correlation plots use a reproducible sample of pairs from the first balanced batch; all quantitative metrics use every within-batch pair.

In [ ]:
DESCRIPTOR_ORDER = ['ECFP_O', 'ECFP_HG', 'COAF', 'DIRECTED_LINEAR']
DESCRIPTOR_LABELS = {
    'ECFP_O': 'ECFP-O',
    'ECFP_HG': 'ECFP-Hg',
    'COAF': 'COAF',
    'DIRECTED_LINEAR': 'Directed linear',
}
DESCRIPTOR_COLORS = {
    'ECFP_O': '#4C78A8',
    'ECFP_HG': '#72B7B2',
    'COAF': '#E45756',
    'DIRECTED_LINEAR': '#F2CF5B',
}
AGGREGATION_LABELS = {
    'arithmetic_mean': 'Arithmetic mean',
    'geometric_mean': 'Geometric mean',
}

def save_figure(fig, filename, png_dpi=600):
    """Save a quantitative figure as editable SVG and high-resolution PNG."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    svg_path = FIGURE_DIR / f'{stem}.svg'
    png_path = FIGURE_DIR / f'{stem}.png'
    fig.savefig(svg_path, format='svg', bbox_inches='tight')
    fig.savefig(png_path, format='png', dpi=png_dpi, bbox_inches='tight')
    print(f'Saved figure: {svg_path}')
    print(f'Saved figure: {png_path}')
    return {'svg': svg_path, 'png': png_path}

def plot_correlation_panels(pair_samples, aggregation='arithmetic_mean'):
    """Plot predicted component similarity against product similarity."""
    subset = pair_samples[pair_samples['aggregation'] == aggregation]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharex=True, sharey=True)
    for ax, descriptor in zip(axes, DESCRIPTOR_ORDER):
        data = subset[subset['descriptor'] == descriptor]
        hb = ax.hexbin(
            data['predicted_component_similarity'],
            data['product_similarity'],
            gridsize=40, mincnt=1, cmap='viridis',
        )
        metric = results.per_batch_metrics.query(
            'descriptor == @descriptor and aggregation == @aggregation and batch == 0'
        ).iloc[0]
        ax.plot([0, 1], [0, 1], '--', color='0.55', linewidth=1)
        ax.set_title(f"{DESCRIPTOR_LABELS[descriptor]}\nSpearman ρ = {metric['spearman']:.3f}")
        ax.set_xlabel('Combined BB similarity')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        fig.colorbar(hb, ax=ax, label='Sampled pair count')
    axes[0].set_ylabel('Assembled-product similarity')
    fig.suptitle(AGGREGATION_LABELS[aggregation], y=1.03)
    fig.tight_layout()
    return fig, axes

def plot_batch_metric_distributions(per_batch_metrics, metric='spearman'):
    """Show descriptor variability across balanced full-library batches."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, aggregation in zip(axes, ['arithmetic_mean', 'geometric_mean']):
        data = per_batch_metrics[per_batch_metrics['aggregation'] == aggregation]
        values = [data.loc[data['descriptor'] == descriptor, metric] for descriptor in DESCRIPTOR_ORDER]
        boxes = ax.boxplot(values, tick_labels=[DESCRIPTOR_LABELS[d] for d in DESCRIPTOR_ORDER], patch_artist=True)
        for patch, descriptor in zip(boxes['boxes'], DESCRIPTOR_ORDER):
            patch.set_facecolor(DESCRIPTOR_COLORS[descriptor])
            patch.set_alpha(0.8)
        ax.set_title(AGGREGATION_LABELS[aggregation])
        ax.tick_params(axis='x', rotation=25)
        ax.grid(axis='y', alpha=0.25)
    axes[0].set_ylabel(metric.replace('_', ' ').title())
    fig.tight_layout()
    return fig, axes

def plot_final_summary(descriptor_summary, aggregation='arithmetic_mean'):
    """Plot mean primary metrics with between-batch SD."""
    subset = descriptor_summary[descriptor_summary['aggregation'] == aggregation]
    metrics = [
        ('spearman', 'Spearman ρ'),
        ('top_5_neighbor_preservation', 'Top-5 preservation'),
        ('top_10_neighbor_preservation', 'Top-10 preservation'),
    ]
    x = np.arange(len(metrics))
    width = 0.19
    fig, ax = plt.subplots(figsize=(10, 4.8))
    for index, descriptor in enumerate(DESCRIPTOR_ORDER):
        row = subset[subset['descriptor'] == descriptor].iloc[0]
        means = [row[f'{metric}_mean'] for metric, _ in metrics]
        errors = [row[f'{metric}_std'] for metric, _ in metrics]
        ax.bar(
            x + (index - 1.5) * width, means, width, yerr=errors, capsize=3,
            color=DESCRIPTOR_COLORS[descriptor], label=DESCRIPTOR_LABELS[descriptor],
        )
    ax.set_xticks(x, [label for _, label in metrics])
    ax.set_ylabel('Mean across balanced batches')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.set_title(AGGREGATION_LABELS[aggregation])
    fig.tight_layout()
    return fig, ax

### Predicted versus observed product similarity

Each panel compares the similarity predicted from the two separate building blocks with the actual similarity of the assembled products. The diagonal indicates numerical agreement; the title reports rank agreement for the complete first batch.

In [ ]:
fig, axes = plot_correlation_panels(results.representative_pair_samples, aggregation='arithmetic_mean')
save_figure(fig, 'predicted_vs_product_similarity_arithmetic_mean.svg')
fig, axes = plot_correlation_panels(results.representative_pair_samples, aggregation='geometric_mean')
save_figure(fig, 'predicted_vs_product_similarity_geometric_mean.svg');

### Robustness across the complete library

The boxplots show the distribution of batch-level Spearman correlations. Narrower distributions indicate that performance is stable across different portions of the DEL rather than being driven by one subset.

In [ ]:
fig, axes = plot_batch_metric_distributions(results.per_batch_metrics, metric='spearman')
save_figure(fig, 'spearman_distributions_across_batches.svg');

## Final results

This table summarizes every balanced batch. `n_products_total` equals the full library size for each descriptor and aggregation. Standard deviations describe variation among batches, not formal confidence intervals.

In [ ]:
display(results.descriptor_summary.round(4))
fig, ax = plot_final_summary(results.descriptor_summary, aggregation='arithmetic_mean')
save_figure(fig, 'descriptor_performance_summary_arithmetic_mean.svg')
fig, ax = plot_final_summary(results.descriptor_summary, aggregation='geometric_mean')
save_figure(fig, 'descriptor_performance_summary_geometric_mean.svg');

## Paired differences relative to ECFP-O

Each difference compares the same batch and aggregation rule. Positive correlation or neighbor-preservation differences favor the named descriptor; negative MAE and RMSE differences indicate lower prediction error than ECFP-O.

In [ ]:
display(results.paired_descriptor_differences.head(20).round(4))

mean_differences = (
    results.paired_descriptor_differences
    .groupby(['descriptor', 'aggregation'], as_index=False)
    .mean(numeric_only=True)
)
display(mean_differences.round(4))